# 02 — Create the late-delivery label

**Job:** define and validate `is_late`. Only delivered orders with both delivery dates
can have an observed outcome. An order is late when its actual customer delivery timestamp
is strictly later than its promised/estimated delivery timestamp.

**Reads:** `artifacts/01_join/ml_orders.csv`.  
**Writes:** `artifacts/02_labels/labeled_orders.csv` and `class_distribution.csv`.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "config.py").exists(): ROOT = ROOT.parent
SOURCE = ROOT / "artifacts" / "01_join" / "ml_orders.csv"
OUT = ROOT / "artifacts" / "02_labels"
OUT.mkdir(parents=True, exist_ok=True)
assert SOURCE.exists(), "Run notebook 01 first"

date_columns = [
    "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
    "order_delivered_customer_date", "order_estimated_delivery_date",
]
orders = pd.read_csv(SOURCE, parse_dates=date_columns)
eligible = orders[
    orders["order_delivered_customer_date"].notna()
    & orders["order_estimated_delivery_date"].notna()
    & orders["order_status"].eq("delivered")
].copy()
eligible["is_late"] = (
    eligible["order_delivered_customer_date"] > eligible["order_estimated_delivery_date"]
).astype("int8")

In [52]:
check = eligible[[
    "order_id", "order_delivered_customer_date", "order_estimated_delivery_date", "is_late"
]].sample(10, random_state=42).copy()
check["manual_check"] = (
    check["order_delivered_customer_date"] > check["order_estimated_delivery_date"]
).astype("int8")
assert check["manual_check"].eq(check["is_late"]).all()
display(check)

distribution = (eligible["is_late"].value_counts().sort_index()
    .rename_axis("is_late").reset_index(name="orders"))
distribution["label"] = distribution["is_late"].map({0: "on_time", 1: "late"})
distribution["share"] = distribution["orders"] / len(eligible)
distribution.to_csv(OUT / "class_distribution.csv", index=False)
display(distribution)

late_share = eligible["is_late"].mean()
print(f"Late share: {late_share:.2%}")
print("Class imbalance exists; accuracy alone would hide poor late-order detection."
      if late_share < 0.30 else "The classes are not severely imbalanced.")
eligible.to_csv(OUT / "labeled_orders.csv", index=False)
print(f"Saved {len(eligible):,} labeled orders")

,order_id,order_delivered_customer_date,order_estimated_delivery_date,is_late,manual_check
9504,c6a73b421eb3e92ce86dbfbbd3530a8f,2018-03-13 17:13:17,2018-03-19,0,0
31123,b132124ca9d69faf63989e08a5851151,2018-06-04 19:26:52,2018-06-08,0,0
27721,7e25a1c58e68fa94300358caad65b944,2017-09-08 17:21:38,2017-09-25,0,0
74022,8418eb39cd68b52032566797b8f1bd11,2017-12-12 21:13:48,2017-12-20,0,0
36187,fe1ec86f91f3b5b6bc46fc3e4b8262cd,2017-12-13 01:16:52,2017-12-15,0,0
74560,c471a28c25283860362c67c0a1cd1694,2018-01-27 01:08:52,2018-02-22,0,0
27653,80e3ea9d65b89f8cbaa3d1f08f5dbd17,2018-06-19 21:50:42,2018-07-26,0,0
8691,a68c330c22c204ef5283a562bed75b97,2018-03-05 18:18:56,2018-03-13,0,0
50743,2728c4c5805dc2b5f0d4aee36cfde5d1,2017-06-19 18:16:09,2017-06-21,0,0
6938,d684202253f5f00621b0afd578646f3d,2017-10-10 20:21:31,2017-10-25,0,0


,is_late,orders,label,share
0,0,88644,on_time,0.918876
1,1,7826,late,0.081124


Late share: 8.11%
Class imbalance exists; accuracy alone would hide poor late-order detection.
Saved 96,470 labeled orders
